In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import re
import time

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    f1_score, accuracy_score, classification_report,
    confusion_matrix
)
from scipy.sparse import hstack, csr_matrix

import textstat
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

print("✅ All imports done")

✅ All imports done


In [2]:
train_df = pd.read_csv('D:/Anas/Fake News Detection/data/processed/train_cleaned.csv')
val_df   = pd.read_csv('D:/Anas/Fake News Detection/data/processed/val_cleaned.csv')
test_df  = pd.read_csv('D:/Anas/Fake News Detection/data/processed/test_cleaned.csv')

for df in [train_df, val_df, test_df]:
    df.dropna(subset=['clean_text'], inplace=True)
    df['clean_text']  = df['clean_text'].astype(str)
    df['statement']   = df['statement'].fillna('').astype(str)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print("✅ Data loaded")

Train: 10240 | Val: 1284 | Test: 1267
✅ Data loaded


In [ ]:
# VADER sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

def extract_features(df):
    features = pd.DataFrame()

    # Feature 1: Statement length (words)
    features['word_count'] = df['statement'].apply(
        lambda x: len(x.split())
    )

    # Feature 2: Character count
    features['char_count'] = df['statement'].apply(len)

    # Feature 3: Punctuation ratio
    features['punct_ratio'] = df['statement'].apply(
        lambda x: len(re.findall(r'[^\w\s]', x)) / max(len(x), 1)
    )

    # Feature 4: Exclamation marks
    features['exclamation'] = df['statement'].apply(
        lambda x: x.count('!')
    )

    # Feature 5: Capital letter ratio
    features['caps_ratio'] = df['statement'].apply(
        lambda x: sum(1 for c in x if c.isupper()) / max(len(x), 1)
    )

    # Feature 6: Flesch Reading Ease Score
    # (high = easy to read, low = complex)
    features['flesch_score'] = df['statement'].apply(
        lambda x: textstat.flesch_reading_ease(x)
    )

    # Feature 7: VADER Sentiment Scores
    features['vader_pos'] = df['statement'].apply(
        lambda x: analyzer.polarity_scores(x)['pos']
    )
    features['vader_neg'] = df['statement'].apply(
        lambda x: analyzer.polarity_scores(x)['neg']
    )
    features['vader_compound'] = df['statement'].apply(
        lambda x: analyzer.polarity_scores(x)['compound']
    )

    # Feature 8: Question marks
    features['question_marks'] = df['statement'].apply(
        lambda x: x.count('?')
    )

    return features

print("Extracting features from train...")
train_hc = extract_features(train_df)

print("Extracting features from val...")
val_hc = extract_features(val_df)

print("Extracting features from test...")
test_hc = extract_features(test_df)

print("\n✅ Handcrafted features extracted!")
print(f"Features per statement: {train_hc.shape[1]}")
print(f"\nSample features (first row):\n{train_hc.iloc[0]}")

Extracting features from train...


In [ ]:
# Step 1: TF-IDF reload (Week 2 wala)
tfidf = joblib.load('D:/Anas/Fake News Detection/models/tfidf_vectorizer.pkl')

X_train_tfidf = tfidf.transform(train_df['clean_text'])
X_val_tfidf   = tfidf.transform(val_df['clean_text'])
X_test_tfidf  = tfidf.transform(test_df['clean_text'])

# Step 2: Handcrafted features ko sparse matrix banao
X_train_hc = csr_matrix(train_hc.values)
X_val_hc   = csr_matrix(val_hc.values)
X_test_hc  = csr_matrix(test_hc.values)

# Step 3: Dono combine karo (hstack)
X_train_combined = hstack([X_train_tfidf, X_train_hc])
X_val_combined   = hstack([X_val_tfidf,   X_val_hc])
X_test_combined  = hstack([X_test_tfidf,  X_test_hc])

# Labels
y_train = train_df['binary_label']
y_val   = val_df['binary_label']
y_test  = test_df['binary_label']

print("✅ Features combined!")
print(f"TF-IDF features   : {X_train_tfidf.shape[1]}")
print(f"Handcrafted feats : {X_train_hc.shape[1]}")
print(f"Combined shape    : {X_train_combined.shape}")

✅ Features combined!
TF-IDF features   : 5000
Handcrafted feats : 10
Combined shape    : (10240, 5010)


In [ ]:
# Model 1: LR with class_weight='balanced'
# (Fake class recall improve karne ke liye)
lr_improved = LogisticRegression(
    max_iter=3000,
    solver       = 'lbfgs',
    random_state=42,
    class_weight='balanced'   # ← YEH NAYA HAI
)
lr_improved.fit(X_train_combined, y_train)

# Model 2: SVM with class_weight='balanced'
svm_improved = LinearSVC(
    max_iter=10000,
    random_state=42,
    class_weight='balanced'   # ← YEH BHI NAYA HAI
)
svm_improved.fit(X_train_combined, y_train)

print("✅ Improved models trained!")

# Save karo
joblib.dump(lr_improved,  'D:/Anas/Fake News Detection/models/lr_improved.pkl')
joblib.dump(svm_improved, 'D:/Anas/Fake News Detection/models/svm_improved.pkl')
print("✅ Saved to models")

✅ Improved models trained!
✅ Saved to models


In [ ]:
def get_scores(model, X, y):
    pred = model.predict(X)
    return {
        'accuracy' : round(accuracy_score(y, pred), 4),
        'macro_f1' : round(f1_score(y, pred, average='macro'), 4),
        'fake_f1'  : round(f1_score(y, pred, average=None)[0], 4),
        'real_f1'  : round(f1_score(y, pred, average=None)[1], 4),
        'fake_rec' : round(f1_score(y, pred, average=None,
                           labels=[0])[0], 4),
    }

# Week 2 models reload
lr_old  = joblib.load('D:/Anas/Fake News Detection/models/logistic_regression.pkl')
svm_old = joblib.load('D:/Anas/Fake News Detection/models/linear_svm.pkl')

results = {
    'Week2 LR (TF-IDF only)'  : get_scores(lr_old,  X_val_tfidf,    y_val),
    'Week3 LR (TF-IDF+HC+Bal)': get_scores(lr_improved, X_val_combined, y_val),
    'Week2 SVM (TF-IDF only)' : get_scores(svm_old, X_val_tfidf,    y_val),
    'Week3 SVM (TF-IDF+HC+Bal)': get_scores(svm_improved, X_val_combined, y_val),
}

print("\n" + "="*65)
print(f"{'Model':<35} {'MacroF1':>8} {'FakeF1':>8} {'RealF1':>8}")
print("="*65)
for name, r in results.items():
    print(f"{name:<35} {r['macro_f1']:>8} {r['fake_f1']:>8} {r['real_f1']:>8}")
print("="*65)


Model                                MacroF1   FakeF1   RealF1
Week2 LR (TF-IDF only)                0.6063   0.5419   0.6707
Week3 LR (TF-IDF+HC+Bal)              0.6096   0.6002    0.619
Week2 SVM (TF-IDF only)               0.6014   0.5673   0.6356
Week3 SVM (TF-IDF+HC+Bal)              0.593   0.5797   0.6063


In [ ]:
# Best model select karo (Macro F1 dekh ke)
best_improved = lr_improved  # (ya svm_improved agar better ho)

y_pred = best_improved.predict(X_test_combined)

print("="*55)
print("  WEEK 3 IMPROVED MODEL — TEST SET RESULTS")
print("="*55)
print(f"  Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"  Macro F1  : {f1_score(y_test, y_pred, average='macro'):.4f}")
print()
print(classification_report(y_test, y_pred,
      target_names=['Fake', 'Real']))
print("="*55)
print(f"\n  Week 2 Baseline (LR) : 0.5973")
print(f"  Week 3 Improved      : "
      f"{f1_score(y_test, y_pred, average='macro'):.4f}")
improved_by = f1_score(y_test, y_pred, average='macro') - 0.5973
print(f"  Change               : {improved_by:+.4f}")

  WEEK 3 IMPROVED MODEL — TEST SET RESULTS
  Accuracy  : 0.6133
  Macro F1  : 0.6085

              precision    recall  f1-score   support

        Fake       0.55      0.58      0.57       553
        Real       0.66      0.64      0.65       714

    accuracy                           0.61      1267
   macro avg       0.61      0.61      0.61      1267
weighted avg       0.62      0.61      0.61      1267


  Week 2 Baseline (LR) : 0.5973
  Week 3 Improved      : 0.6085
  Change               : +0.0112
